In [1]:
import re
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

from collections import Counter
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

In [2]:
categories = ["sci.space", "rec.sport.baseball"]

data = fetch_20newsgroups(
    subset="all",
    categories=categories,
    remove=("headers", "footers", "quotes")
)

texts = data.data
labels = data.target

print(len(texts))
print(categories)
print(texts[0][:300])

1981
['sci.space', 'rec.sport.baseball']

Do you really have *that* much information on him?  Really?


I don't know.  You tell me.  What percentage of players reach or 
exceed their MLE's *in their rookie season*?  We're talking about
1993, you know.


If that were your purpose, maybe.  Offerman spent 1992 getting 
acclimated, if you will


In [3]:
def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return text.split()

tokens = tokenize("BERT learns contextual word representations")
print(tokens)

['bert', 'learns', 'contextual', 'word', 'representations']


In [4]:
PAD = "[PAD]"
UNK = "[UNK]"
CLS = "[CLS]"
SEP = "[SEP]"

counter = Counter()

for text in texts:
    counter.update(tokenize(text))

vocab_size = 8000

vocab = {
    PAD: 0,
    UNK: 1,
    CLS: 2,
    SEP: 3
}

for word, freq in counter.most_common(vocab_size - 4):
    vocab[word] = len(vocab)

print("Vocab size:", len(vocab))

Vocab size: 8000


In [5]:
max_len = 128

def encode_text(text):
    tokens = [CLS] + tokenize(text)[:max_len - 2] + [SEP]

    ids = [vocab.get(tok, vocab[UNK]) for tok in tokens]

    attention_mask = [1] * len(ids)

    while len(ids) < max_len:
        ids.append(vocab[PAD])
        attention_mask.append(0)

    return torch.tensor(ids), torch.tensor(attention_mask)

In [6]:
class NewsDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
       
        return len(self.texts)

    def __getitem__(self, idx):
        input_ids, attention_mask = encode_text(self.texts[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.long)

        return input_ids, attention_mask, label

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

train_dataset = NewsDataset(X_train, y_train)
test_dataset = NewsDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

class BertEmbedding(nn.Module):
def 

In [8]:
class BertEmbeddings(nn.Module):
       def __init__(self,vocab_size,embed_dim,max_len):
              super().__init__()
              self.token_embeddings = nn.Embedding(vocab_size, embed_dim)
              self.position_embeddings=nn.Embedding(max_len, embed_dim)

              def forward(self,input_ids):
                     batch_size,seq_len=input_ids.shape
                     positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).expand(batch_size, -1)
                     positions=positions.to(input_ids.device)

                     token_emb=self.token_embeddings(input_ids)
                     pos_emb=self.position_embeddings(positions)
                     return token_emb+pos_emb

multi head attention 

In [9]:
class multiheadselfattention(nn.Module):
       def __init__(self,embed_dim,num_heads):
              super().__init__()
              assert embed_dim % num_heads==0 
              self.embed_dim = embed_dim
              self.num_heads = num_heads
              self.head_dim = embed_dim // num_heads
              self.q_linear = nn.Linear(embed_dim, embed_dim)
              self.k_linear=nn.Linear(embed_dim, embed_dim)
              self.v_linear=nn.Linear(embed_dim, embed_dim)
              self.out=nn.Linear(embed_dim, self.embed_dim)

       def forward(self,x,attention_mask):
              B,S,E=x.shape
              Q=self.q_linear(x).view(B,S,self.num_heads,self.head_dim).transpose(1,2)
              K=self.k_linear(x).view(B,S,self.num_heads,self.head_dim).transpose(2,1)
              V=self.v_linear(x).view(B,S,self.num_heads,self.head_dim).transpose(1,2)
              scores=torch.matmul(Q,K.transpose(-2,-1))/math.sqrt(self.head_dim)
              scores=scores.masked_fill(attention_mask.unsqueeze(1).unsqueeze(2)==0,float("-inf"))
              mask=attention_mask.unsqueeze(1).unsqueeze(2)
              scores=scores.masked_fill(mask==0,-1e9)

              attention_weights=F.softmax(scores,dim=-1)
              context=torch.matmul(attention_weights,V)
              context=context.transpose(1,2).contiguous().view(B,S,E)
              context=context.view(B,S,E)
              return self.out(context)

feef forwad network 

In [11]:
class feedforward(nn.Module):
       def __init__(self,embed_dim,hidden_dim):
              super().__init__()
              self.fc1=nn.Linear(embed_dim,hidden_dim)
              self.fc2=nn.Linear(hidden_dim,embed_dim)

       def forward(self,x):
              x=self.fc1(x)
              x=F.gelu(x)
              x=self.fc2(x)
              return x

transformer encoder block

In [13]:
class TransformerEncoderBlock(nn.Module):
       def __init__(self,embed_dim,num_heads,ff_hidden_dim,dropout=0.1):
              super().__init__()

              self.attention=multiheadselfattention(embed_dim,num_heads)
              self.norm1=nn.LayerNorm(embed_dim)
              self.ff=feedforward(embed_dim,ff_hidden_dim)
              self.norm2=nn.LayerNorm(embed_dim)
              self.dropout=nn.Dropout(dropout)
       def forward(self,x,attention_mask):
              attn_output=self.attention(x,attention_mask)
              x=self.norm1(x+self.dropout(attn_output))
              ff_output=self.ff(x)
              x=self.norm2(x+self.dropout(ff_output))
              return x

full bert model

In [14]:
# =========================
# 12. Mini-BERT Classifier
# =========================
class MiniBERTClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        max_len,
        num_classes,
        embed_dim=128,
        num_heads=4,
        ff_hidden_dim=256,
        num_layers=2,
        dropout=0.1
    ):
        super().__init__()

        self.embedding = BertEmbedding(vocab_size, embed_dim, max_len)

        self.encoder_layers = nn.ModuleList([
            TransformerEncoderBlock(
                embed_dim,
                num_heads,
                ff_hidden_dim,
                dropout
            )
            for _ in range(num_layers)
        ])

        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, input_ids, attention_mask):
        x = self.embedding(input_ids)

        for layer in self.encoder_layers:
            x = layer(x, attention_mask)

        cls_output = x[:, 0, :]

        logits = self.classifier(cls_output)

        return logits

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MiniBERTClassifier(
    vocab_size=len(vocab),
    max_len=max_len,
    num_classes=2
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
criterion = nn.CrossEntropyLoss()

print(model)

NameError: name 'BertEmbedding' is not defined